In [1]:
import logging

import numpy as np
import optimistix as optx
from atmodeller import (
    EquilibriumModel,
    Planet,
    ChemicalSpecies,
    SpeciesNetwork,
    debug_logger,
    earth_oceans_to_hydrogen_mass,
    SolverParameters,
)
from atmodeller.eos import get_eos_models
from atmodeller.solubility import get_solubility_models
from atmodeller.thermodata import IronWustiteBuffer
from atmodeller.interfaces import FugacityConstraintProtocol, ThermodynamicStateProtocol
from atmodeller import debug_logger
from atmodeller.classes import EquilibriumModel
from atmodeller.containers import (
    ChemicalSpecies,
    FixedFugacityConstraint,
    SpeciesNetwork,
    ThermodynamicState,
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import LogNorm
from matplotlib.patches import Patch

from scipy.interpolate import interp1d
from scipy.optimize import fsolve, root_scalar
import os

from matplotlib.ticker import ScalarFormatter

formatter = ScalarFormatter(useMathText=False)
formatter.set_scientific(False)
formatter.set_useOffset(False)
from mpl_toolkits.axes_grid1.axes_divider import make_axes_locatable

logger = debug_logger() #you can dump it in file 
#logger.setLevel(logging.INFO) #you could set it to error

logger.setLevel(logging.DEBUG)  

#gamma 0.05 0.01 guillot 2010 f 1/4 tau compute 

Atmodeller initialized with double precision (float64)


# Definitions (run these first)

In [2]:
color_tot = "black"
color_H2 = "orange"
color_H2O = "blue"
color_SiH4 = "red"
color_SiO2 = "yellow"
color_SiO = "brown"
color_O2 = "limegreen"
color_CO2 = "cyan"
color_CO = "magenta"
color_CH4 = "purple"
color_N2 = "green"
color_NH3 = "pink"
color_MgO = "darkred"
color_He = "gray"
color_H = "orange"
color_C = "cyan"
color_N = "green"
color_O = "blue"
color_Si = "red"
color_Mg = "lime"
color_MgH2 = "gold"
color_MgH = "teal"
color_MgOH = "indigo"
color_Ccr = "dimgray"

In [3]:
SiO2_mantlemasspercent_palme14: float = 45.4
total_mantlemasspercent_palme14: float = 98.41
core_mass_fraction: float = 0.327  # typical values used are between 0.325 - 0.33
Si_massfraction_palme14: float = round(
    SiO2_mantlemasspercent_palme14
    / total_mantlemasspercent_palme14
    * (1 - core_mass_fraction)
    / (28.0855 + 2 * 15.999)
    * 28.0855,
    3,
)

O_massfraction_palme14: float = Si_massfraction_palme14 / 28.0855 * 2 * 15.999
# Lodders et al. (2009) Springer Book Chapter - Table 8 (in wt%)

H_masspercent_lodders09: float = 73.9
He_masspercent_lodders09: float = 24.69
C_masspercent_lodders09: float = 0.22
N_masspercent_lodders09: float = 0.07
O_masspercent_lodders09: float = 0.63
Si_masspercent_lodders09: float = 0.07
Mg_masspercent_lodders09: float = 0.06

# Lodders et al. (2009) Springer Book Chapter - Table 6 (log-normalalized abundances relative to H)

H_logN: float = 12
He_logN: float = 10.93
C_logN: float = 8.39
N_logN: float = 7.86
O_logN: float = 8.73
Si_logN: float = 7.53
Mg_logN: float = 7.54 

In [4]:
eos_models = get_eos_models()
sol_models = get_solubility_models()

H2O_g = ChemicalSpecies.create_gas("H2O")
H2O_gs = ChemicalSpecies.create_gas("H2O", solubility=sol_models["H2O_peridotite_sossi23"])
H2O_rgs = ChemicalSpecies.create_gas(
    "H2O",
    activity=eos_models["H2O_cork_holland98"],
    solubility=sol_models["H2O_peridotite_sossi23"],
)
H2O_rg = ChemicalSpecies.create_gas(
    "H2O",
    activity=eos_models["H2O_cork_holland98"],
)

H2_g = ChemicalSpecies.create_gas("H2")
H2_gs = ChemicalSpecies.create_gas("H2", solubility=sol_models["H2_basalt_hirschmann12"])
H2_rgs = ChemicalSpecies.create_gas(
    "H2", activity=eos_models["H2_chabrier21"], solubility=sol_models["H2_basalt_hirschmann12"]
)
H2_rg = ChemicalSpecies.create_gas(
    "H2", activity=eos_models["H2_chabrier21"]
)

O2_g = ChemicalSpecies.create_gas("O2")
O2_rg = ChemicalSpecies.create_gas("O2", activity=eos_models["O2_cs_shi92"])

OSi_g = ChemicalSpecies.create_gas("OSi")
OSi_rg = ChemicalSpecies.create_gas("OSi", activity=eos_models["OSi_rk49_connolly16"])

H4Si_g = ChemicalSpecies.create_gas("H4Si")
H4Si_rg = ChemicalSpecies.create_gas("H4Si", activity=eos_models["H4Si_wang18"])

O2Si_cd = ChemicalSpecies.create_condensed("O2Si", state="cd")

C_cd = ChemicalSpecies.create_condensed("C", state="cd")
CSi_cd = ChemicalSpecies.create_condensed("CSi", state="cd")
Si_cd = ChemicalSpecies.create_condensed("Si", state="cd")
N4Si3_cd = ChemicalSpecies.create_condensed("N4Si3", state="cd")
H2O_cd = ChemicalSpecies.create_condensed("H2O", state="cd")

CO2_g = ChemicalSpecies.create_gas("CO2")
CO2_gs = ChemicalSpecies.create_gas("CO2", solubility=sol_models["CO2_basalt_dixon95"])
CO2_rgs = ChemicalSpecies.create_gas(
    "CO2", solubility=sol_models["CO2_basalt_dixon95"], activity=eos_models["CO2_cs_shi92"]
)
CO2_rg = ChemicalSpecies.create_gas(
    "CO2",activity=eos_models["CO2_cs_shi92"]
)

CO_g = ChemicalSpecies.create_gas("CO")
CO_gs = ChemicalSpecies.create_gas("CO", solubility=sol_models["CO_basalt_yoshioka19"])
CO_rgs = ChemicalSpecies.create_gas(
    "CO", solubility=sol_models["CO_basalt_yoshioka19"], activity=eos_models["CO_cs_shi92"]
)
CO_rg = ChemicalSpecies.create_gas(
    "CO", activity=eos_models["CO_cs_shi92"]
)

CH4_g = ChemicalSpecies.create_gas("CH4")
CH4_gs = ChemicalSpecies.create_gas("CH4", solubility=sol_models["CH4_basalt_ardia13"])
CH4_rgs = ChemicalSpecies.create_gas(
    "CH4", solubility=sol_models["CH4_basalt_ardia13"], activity=eos_models["CH4_cs_shi92"]
)
CH4_rg = ChemicalSpecies.create_gas(
    "CH4", activity=eos_models["CH4_cs_shi92"]
)

N2_g = ChemicalSpecies.create_gas("N2")
N2_gs = ChemicalSpecies.create_gas("N2", solubility=sol_models["N2_basalt_libourel03"])
N2_rgs = ChemicalSpecies.create_gas(
    "N2",solubility=sol_models["N2_basalt_libourel03"], activity=eos_models["N2_cs_saxena87"]
)
N2_rg = ChemicalSpecies.create_gas(
    "N2", activity=eos_models["N2_cs_saxena87"]
)


NH3_g = ChemicalSpecies.create_gas("H3N")
NH3_rg = ChemicalSpecies.create_gas("H3N", activity=eos_models["H3N_rk49_reid87"])

He_g = ChemicalSpecies.create_gas("He")
He_gs = ChemicalSpecies.create_gas("He", solubility=sol_models["He_basalt_jambon86"])
He_rgs = ChemicalSpecies.create_gas(
    "He", solubility=sol_models["He_basalt_jambon86"], activity=eos_models["He_chabrier21"]
)
He_rg = ChemicalSpecies.create_gas(
    "He", activity=eos_models["He_chabrier21"]
)


species_HOSi_magma_nosol_ideal = SpeciesNetwork((H2O_g, H2_g, O2_g, OSi_g, H4Si_g, O2Si_cd))
species_HOSi_magma_sol_ideal = SpeciesNetwork((H2O_gs, H2_gs, O2_g, OSi_g, H4Si_g, O2Si_cd))
species_HOSi_magma_sol_real = SpeciesNetwork((H2O_rgs, H2_rgs, O2_rg, OSi_rg, H4Si_rg, O2Si_cd))
species_HOSi_atm_sol_real = SpeciesNetwork((H2O_g, H2_g, O2_g, H2O_cd,OSi_rg, H4Si_rg))


species_HHeCNOSi_magma_nosol_ideal = SpeciesNetwork(
    (H2O_g, H2_g, O2_g, OSi_g, H4Si_g, O2Si_cd, CO2_g, CO_g, CH4_g, N2_g, NH3_g, He_g)
)
species_HHeCNOSi_magma_sol_ideal = SpeciesNetwork(
    (H2O_gs, H2_gs, O2_g, OSi_g, H4Si_g, O2Si_cd, CO2_gs, CO_gs, CH4_gs, N2_gs, NH3_g, He_gs)
)
species_HHeCNOSi_magma_sol_real = SpeciesNetwork(
    (
        H2O_rgs,
        H2_rgs,
        O2_rg,
        OSi_rg,
        H4Si_rg,
        O2Si_cd,
        CO2_rgs,
        CO_rgs,
        CH4_rgs,
        N2_rgs,
        NH3_rg,
        He_rgs,
    )
)

species_HHeCNOSi_atmosphere = SpeciesNetwork(
    (H2O_rg, H2_rg, O2_rg, OSi_rg, H4Si_rg, H2O_cd, C_cd, CSi_cd, Si_cd, N4Si3_cd,
     CO2_rg, CO_rg, CH4_rg, N2_rg, NH3_rg, He_rgs)
)
#species_HHeCNOSi_atmosphere = SpeciesNetwork(
#    (H2O_g, H2_g, O2_g, OSi_g, H4Si_g, O2Si_cd, C_cd, CSi_cd, Si_cd, N4Si3_cd,
#     CO2_g, CO_g, CH4_g, N2_g, NH3_g, He_g)
#)

In [5]:
number_of_realisations = 100

# Overall structure 
R_earth = 6.371e6
R_uranus = 3.939 * R_earth
M_earth = 5.972e24 #kg
M_uranus = 14.536 * M_earth

#fO2_log10_shifts = [-1]
fO2_log10_shifts = np.linspace(-6, 0, num=30)  # IW-6 to IW
fO2_log10_shifts = [-6,-5,-4,-3,-2,-1,0]
hmps = np.logspace(-1,0, num=50)  # wt% H
h_kgs = hmps / 100 * M_uranus  # kg

si_kgs_solar = h_kgs * Si_masspercent_lodders09 / H_masspercent_lodders09
mg_kgs_solar = h_kgs * Mg_masspercent_lodders09 / H_masspercent_lodders09
o_kgs_solar = h_kgs * O_masspercent_lodders09 / H_masspercent_lodders09
c_kgs_solar = h_kgs * C_masspercent_lodders09 / H_masspercent_lodders09
n_kgs_solar = h_kgs * N_masspercent_lodders09 / H_masspercent_lodders09
he_kgs_solar = h_kgs * He_masspercent_lodders09 / H_masspercent_lodders09


rho_centres = [2500,3000,2000,1500,1000]
rho_centres = [3000]
rho_centre = 3000
print(rho_centres)
R_core = 0.7*R_uranus
Lowerbound_M_core = [
    4*np.pi*(R_core**3)*rho/3
    for rho in rho_centres
]
M_cores = Lowerbound_M_core
print(M_cores)
print(10*M_earth)

[3000]
[6.812158460855618e+25]
5.972000000000001e+25


In [6]:
f = 1/4

T_solar = 5772
R_solar = 6.957e8
d_solar_Uranus = 2876679082000

T_irr = T_solar * np.sqrt(R_solar/d_solar_Uranus)

T_eff = 59.1 #pm 0.3 The albedo, effective temperature, and energy balance of Uranus, as determined from Voyager IRIS data
T_eq = 58.2 #pm 1 The albedo, effective temperature, and energy balance of Uranus, as determined from Voyager IRIS data
T_int = (T_eff**(4) - T_eq**(4))**(1/4)
T_int = 120


gamma_1 = 0.01
gamma_2 = 0.05
gamma_3 = 0.1
gammas = np.logspace(-4,2,7)
gamma = gamma_2
tau = np.logspace(-10,10,1000)

In [7]:
def Temp_profile(tau):
    return (((3*T_int**(4))/4)*((2/3)+tau) + ((3*f*T_irr**(4))/4)*((2/3)+ (1/(gamma*np.sqrt(3))) + ((gamma/(np.sqrt(3)))- (1/(gamma*np.sqrt(3))))*np.exp(-gamma*tau*np.sqrt(3))))**(1/4)  

def find_tau(tau, target, initial_guess):
    def g(tau):
        return Temp_profile(tau) - target
    x_solution = fsolve(g, initial_guess)
    return x_solution[0]
def find_scale(Temperature, Pressure, tau, initial_guess):
    tau_scale = find_tau(tau,Temperature, initial_guess)
    scale_factor = Pressure/tau_scale
    return scale_factor

def Pressure_Temperatures(T_original, tau, Pressure, initial_guess):
    scales = find_scale(T_original, Pressure, tau, initial_guess)
    Pressures = tau*scales
    Temperatures = Temp_profile(tau)
    indices = [i for i, v in enumerate(Temperatures) if 100 < v < 6000]
    return Pressures[indices], Temperatures[indices]


In [8]:
def create_phase_lookup(filename):
    """
    Create a simple phase lookup function
    """
    # Load data
    data = []
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                vals = line.split()
                float(vals[0])  # Test if numeric
                data.append([float(x) for x in vals[:11]])
            except (ValueError, IndexError):
                continue
    
    data = np.array(data)
    
    # Phase names
    phase_names = {
        -1: "Ice Ih", -2: "Ice II", -3: "Ice III",
        -5: "Ice V", -6: "Ice VI", -7: "Ice VII",
        3: "Vapor", 4: "Liquid", 5: "Supercritical"
    }
    
    # Extract columns
    temps = data[:, 1]      # Temperature in K
    press = data[:, 0] / 1e9  # Pressure in GPa
    phases = data[:, 10]     # Phase ID
    
    def lookup(T, P):
        """Return phase name at T(K) and P(GPa)"""
        # Find closest point
        distances = (temps - T)**2 + (press - P)**2
        idx = np.argmin(distances)
        phase_id = int(phases[idx])
        return phase_names.get(phase_id, f"Phase {phase_id}")
    
    return lookup

filename = '/home/mathias/H2Oeq/AQUA/Tables/aqua_eos_pt_v1_0.dat'
get_phase = create_phase_lookup(filename)

In [9]:
def read_aqua_simple(filename):
    """
    Read AQUA data file, skipping header lines that start with text
    """
    data = []
    line_count = 0
    skipped_count = 0
    
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            line_count += 1
            
            # Skip empty lines
            if not line:
                continue
            
            # Try to convert to float - if it fails, skip the line
            try:
                # Split and try to convert first element to float
                vals = line.split()
                if vals:  # Check if line has content
                    float(vals[0])  # Test if first value is a number
                    # If successful, convert all values to float
                    data.append([float(x) for x in vals[:11]])
            except ValueError:
                # This line contains text - skip it
                skipped_count += 1
                continue
    
    print(f"Read {line_count} total lines")
    print(f"Skipped {skipped_count} header/comment lines")
    print(f"Loaded {len(data)} data points")
    
    if len(data) == 0:
        raise ValueError("No data points were loaded! Check file format.")
    
    # Convert to DataFrame
    df = pd.DataFrame(data, columns=[
        'press_Pa', 'temp_K', 'rho', 'ad_grad', 's', 'u', 
        'c', 'mmw', 'x_ion', 'x_d', 'phase_id'
    ])
    
    # Convert pressure to GPa for convenience
    df['press_GPa'] = df['press_Pa'] / 1e9
    
    # Ensure phase_id is integer
    df['phase_id'] = df['phase_id'].astype(int)
    
    # Phase names mapping
    phase_names = {
        -1: "Ice Ih",
        -2: "Ice II",
        -3: "Ice III",
        -5: "Ice V",
        -6: "Ice VI",
        -7: "Ice VII",
        -10: "Ice X",
        3: "Vapor",
        4: "Liquid",
        5: "Supercritical/Superionic"
    }
    
    def get_phase(T, P):
        """Get phase at temperature T(K) and pressure P(GPa)"""
        # Find closest point
        distances = np.sqrt(
            (df['temp_K'] - T)**2 + 
            (df['press_GPa'] - P)**2
        )
        idx = distances.idxmin()
        pid = df.loc[idx, 'phase_id']
        
        # Also return density and actual T,P for verification
        actual_T = df.loc[idx, 'temp_K']
        actual_P = df.loc[idx, 'press_GPa']
        density = df.loc[idx, 'rho']
        
        phase_name = phase_names.get(pid, f"Unknown Phase ({pid})")
        
        return {
            'phase': phase_name,
            'phase_id': pid,
            'density': density,
            'actual_T': actual_T,
            'actual_P': actual_P,
            'distance': distances[idx]
        }
    
    # Print summary
    print(f"\n✅ Successfully loaded {len(df):,} data points")
    print(f"   Temperature range: {df['temp_K'].min():.0f} - {df['temp_K'].max():.0f} K")
    print(f"   Pressure range: {df['press_GPa'].min():.6f} - {df['press_GPa'].max():.2f} GPa")
    print(f"   Phase IDs present: {sorted(df['phase_id'].unique())}")
    
    return df, get_phase

# Alternative: Using pandas with comment handling
def read_aqua_pandas(filename):
    """
    Alternative method using pandas with comment handling
    """
    # Try to find where the numeric data starts
    with open(filename, 'r') as f:
        for i, line in enumerate(f):
            try:
                # Try to parse first few numbers
                vals = line.strip().split()
                if vals:
                    float(vals[0])
                    float(vals[1])
                    # If successful, this is the start of data
                    skip_rows = i
                    print(f"Data starts at line {i+1}")
                    break
            except (ValueError, IndexError):
                continue
    
    # Read with pandas, skipping header rows
    df = pd.read_csv(
        filename,
        sep=r'\s+',
        skiprows=skip_rows,
        names=['press_Pa', 'temp_K', 'rho', 'ad_grad', 's', 'u', 
               'c', 'mmw', 'x_ion', 'x_d', 'phase_id'],
        engine='python'
    )
    
    df['press_GPa'] = df['press_Pa'] / 1e9
    df['phase_id'] = df['phase_id'].astype(int)
    
    return df

# ===== USAGE =====

filename = '/home/mathias/H2Oeq/AQUA/Tables/aqua_eos_pt_v1_0.dat'

# Method 1: Simple function
print("="*60)
print("LOADING AQUA DATA")
print("="*60)
df, get_phase = read_aqua_simple(filename)

# Test points
print("\n" + "="*60)
print("TEST POINTS")
print("="*60)

test_points = [
    (300, 0.0001),   # Near surface
    (500, 1),         # 1 GPa
    (1000, 10),       # 10 GPa
    (2000, 50),       # 50 GPa
    (4000, 100),      # 100 GPa
]

for T, P in test_points:
    result = get_phase(T, P)
    print(f"T={T:4.0f}K, P={P:6.2f}GPa → {result['phase']:20s} "
          f"(ρ={result['density']:8.1f} kg/m³) "
          f"[closest point: T={result['actual_T']:.0f}K, P={result['actual_P']:.3f}GPa]")

# Quick lookup function for single use
def quick_phase(T, P):
    """Quick one-line phase lookup"""
    return get_phase(T, P)['phase']


LOADING AQUA DATA
Read 329012 total lines
Skipped 19 header/comment lines
Loaded 328993 data points

✅ Successfully loaded 328,993 data points
   Temperature range: 100 - 100000 K
   Pressure range: 0.000000 - 400000.00 GPa
   Phase IDs present: [np.int64(-10), np.int64(-7), np.int64(-6), np.int64(-5), np.int64(-3), np.int64(-2), np.int64(-1), np.int64(3), np.int64(4), np.int64(5)]

TEST POINTS
T= 300K, P=  0.00GPa → Liquid               (ρ=   996.0 kg/m³) [closest point: T=302K, P=0.000GPa]
T= 500K, P=  1.00GPa → Liquid               (ρ=  1139.4 kg/m³) [closest point: T=501K, P=1.003GPa]
T=1000K, P= 10.00GPa → Supercritical/Superionic (ρ=  1618.3 kg/m³) [closest point: T=1000K, P=10.034GPa]
T=2000K, P= 50.00GPa → Supercritical/Superionic (ρ=  2338.4 kg/m³) [closest point: T=1995K, P=50.297GPa]
T=4000K, P=100.00GPa → Supercritical/Superionic (ρ=  2724.1 kg/m³) [closest point: T=3981K, P=100.366GPa]


In [11]:
def list_phases(Temperatures, Pressures):
    #pressures in Gpa, Temperatures in kelvin
    phases = []
    for i in range(len(Temperatures)):
        phase = quick_phase(Temperatures[i], Pressures[i])
        phases.append(phase)
        phases = list(set(phases))
    return phases

# Simulation magma-core boundary

In [12]:
# Initialize EquilibriumModel objects 
model_magma_nosol_ideal = EquilibriumModel(species_HOSi_magma_nosol_ideal)
model_magma_sol_ideal = EquilibriumModel(species_HOSi_magma_sol_ideal)
model_magma_sol_real = EquilibriumModel(species_HOSi_magma_sol_real)

number_of_realisations = 50
mantle_melt_fraction = 1
init_metallicitys = [1]
for i in range(len(rho_centres)):
    rho_centre = rho_centres[i]
    M_core = M_cores[i]
    fugacity_constraints = {O2_rg.name: IronWustiteBuffer(fO2_log10_shifts)}
    for fO2_log10_shift in fO2_log10_shifts:
        shift = f"{fO2_log10_shift:.2g}"
        planet = Planet(
            temperature=3000,
            planet_mass=M_core,
            mantle_melt_fraction=1,
            surface_radius=0.5*R_uranus,
        )
        si_kg_magma = Si_massfraction_palme14 * M_core
        for init_metallicity in init_metallicitys:
            mass_constraints = {
                "H": h_kgs,
                "Si": init_metallicity * si_kgs_solar + mantle_melt_fraction * si_kg_magma,
            }
            # Magma - No solubility - Ideal Gas
            model_magma_nosol_ideal.solve(
                state=planet,
                mass_constraints=mass_constraints,
                fugacity_constraints={O2_rg.name: IronWustiteBuffer(np.full(number_of_realisations, fO2_log10_shift))},
                solver_parameters=SolverParameters(multistart=20)
            )
            output_magma_nosol_ideal = model_magma_nosol_ideal.output
            # output_magma_nosol_ideal.quick_look()
            output_magma_nosol_ideal.to_excel(
                f"./excel_Uranus_solar_3000K/HOSi/rho{rho_centre}/core_boundary/HOSi_magma_nosol_ideal_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_TEST"
            )
            output_magma_nosol_ideal.to_pickle(
                f"./excel_Uranus_solar_3000K/HOSi/rho{rho_centre}/core_boundary/HOSi_magma_nosol_ideal_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_TEST"
            )
    
            # Magma - Solubility - Ideal Gas
            initial_log_number_moles = output_magma_nosol_ideal.log_number_moles
            model_magma_sol_ideal.solve(
                state=planet,
                mass_constraints=mass_constraints,
                initial_log_number_moles=initial_log_number_moles,
                fugacity_constraints={O2_rg.name: IronWustiteBuffer(np.full(number_of_realisations, fO2_log10_shift))},
                solver_parameters=SolverParameters(multistart=20)
            )
            output_magma_sol_ideal = model_magma_sol_ideal.output
            # output_magma_sol_ideal.quick_look()
            output_magma_sol_ideal.to_excel(
                f"./excel_Uranus_solar_3000K/HOSi/rho{rho_centre}/core_boundary/HOSi_magma_sol_ideal_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_TEST"
            )
            output_magma_sol_ideal.to_pickle(
                f"./excel_Uranus_solar_3000K/HOSi/rho{rho_centre}/core_boundary/HOSi_magma_sol_ideal_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_TEST"
            )
            # Magma - Solubility - Real Gas
            initial_log_number_moles = output_magma_sol_ideal.log_number_moles
            model_magma_sol_real.solve(
                state=planet,
                mass_constraints=mass_constraints,
                initial_log_number_moles=initial_log_number_moles,
                fugacity_constraints={O2_rg.name: IronWustiteBuffer(np.full(number_of_realisations, fO2_log10_shift))},
                solver_parameters=SolverParameters(multistart=50)
            )
            output_magma_sol_real = model_magma_sol_real.output
            output_magma_sol_real.quick_look()
            output_magma_sol_real.to_excel(
                f"./excel_Uranus_solar_3000K/HOSi/rho{rho_centre}/core_boundary/HOSi_magma_sol_real_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_TEST"
            )
            output_magma_sol_real.to_pickle(
                f"./excel_Uranus_solar_3000K/HOSi/rho{rho_centre}/core_boundary/HOSi_magma_sol_real_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_TEST"
            )

[22:51:43 - atmodeller.classes             - INFO     ] - species_network = ('H2O_g: IdealGas, NoSolubility', 'H2_g: IdealGas, NoSolubility', 'O2_g: IdealGas, NoSolubility', 'OSi_g: IdealGas, NoSolubility', 'H4Si_g: IdealGas, NoSolubility', 'O2Si_cd: CondensateActivity, NoSolubility')
[22:51:43 - atmodeller.classes             - INFO     ] - Thermodynamic data requires temperatures between 200 K and 6000 K
[22:51:43 - atmodeller.classes             - INFO     ] - reactions = {0: '2.0 H2O_g = 2.0 H2_g + 1.0 O2_g',
 1: '3.0 H2_g + 1.0 OSi_g = 1.0 H2O_g + 1.0 H4Si_g',
 2: '1.0 H2O_g + 1.0 OSi_g = 1.0 H2_g + 1.0 O2Si_cd'}
[22:51:43 - atmodeller.classes             - INFO     ] - species_network = ('H2O_g: IdealGas, SolubilityPowerLaw', 'H2_g: IdealGas, SolubilityPowerLawLog10', 'O2_g: IdealGas, NoSolubility', 'OSi_g: IdealGas, NoSolubility', 'H4Si_g: IdealGas, NoSolubility', 'O2Si_cd: CondensateActivity, NoSolubility')
[22:51:43 - atmodeller.classes             - INFO     ] - Thermodynamic

In [18]:
init_metallicity = 1
mantle_melt_fraction = 1
for fO2_log10_shift in fO2_log10_shifts:
    shift = f"{fO2_log10_shift:.2g}"
    for i in range(1):
        hmp = hmps[0]
        filename = f"./excel_Uranus_solar_3000K/HOSi/rho{rho_centre}/core_boundary/HOSi_magma_sol_real_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_2.xlsx"
        Pressure = pd.read_excel(filename, sheet_name="state")["pressure"]
        Pressures, Temperatures = Pressure_Temperatures(3000, tau, Pressure[i], 1e7)
        Pressures = Pressures*10**(-4)
        for i in range(len(Temperatures)):
            result = get_phase(Temperatures[i], Pressures[i])
            print(f"{result['phase']:20s} "
              f"(ρ={result['density']:8.1f} kg/m³) "
              f"[closest point: T={result['actual_T']:.0f}K, P={result['actual_P']:.3f}GPa]")
        print(list_phases(Temperatures, Pressures))

/tmp/ipykernel_957/2124516973.py:2: RuntimeWarning: overflow encountered in exp
  return (((3*T_int**(4))/4)*((2/3)+tau) + ((3*f*T_irr**(4))/4)*((2/3)+ (1/(gamma*np.sqrt(3))) + ((gamma/(np.sqrt(3)))- (1/(gamma*np.sqrt(3))))*np.exp(-gamma*tau*np.sqrt(3))))**(1/4)


Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih      

/tmp/ipykernel_957/2124516973.py:2: RuntimeWarning: overflow encountered in exp
  return (((3*T_int**(4))/4)*((2/3)+tau) + ((3*f*T_irr**(4))/4)*((2/3)+ (1/(gamma*np.sqrt(3))) + ((gamma/(np.sqrt(3)))- (1/(gamma*np.sqrt(3))))*np.exp(-gamma*tau*np.sqrt(3))))**(1/4)


Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih      

/tmp/ipykernel_957/2124516973.py:2: RuntimeWarning: overflow encountered in exp
  return (((3*T_int**(4))/4)*((2/3)+tau) + ((3*f*T_irr**(4))/4)*((2/3)+ (1/(gamma*np.sqrt(3))) + ((gamma/(np.sqrt(3)))- (1/(gamma*np.sqrt(3))))*np.exp(-gamma*tau*np.sqrt(3))))**(1/4)


Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih      

... (4 outputs left out)

/tmp/ipykernel_957/2124516973.py:2: RuntimeWarning: overflow encountered in exp
  return (((3*T_int**(4))/4)*((2/3)+tau) + ((3*f*T_irr**(4))/4)*((2/3)+ (1/(gamma*np.sqrt(3))) + ((gamma/(np.sqrt(3)))- (1/(gamma*np.sqrt(3))))*np.exp(-gamma*tau*np.sqrt(3))))**(1/4)


Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih      

/tmp/ipykernel_957/2124516973.py:2: RuntimeWarning: overflow encountered in exp
  return (((3*T_int**(4))/4)*((2/3)+tau) + ((3*f*T_irr**(4))/4)*((2/3)+ (1/(gamma*np.sqrt(3))) + ((gamma/(np.sqrt(3)))- (1/(gamma*np.sqrt(3))))*np.exp(-gamma*tau*np.sqrt(3))))**(1/4)


Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=102K, P=0.000GPa]
Ice Ih      

# Simulating atmosphere

In [17]:
model_atm_magma_sol_real = EquilibriumModel(species_HOSi_atm_sol_real)
init_metallicity = 1
mantle_melt_fraction = 1
for i in range(len(rho_centres)):
    rho_centre = rho_centres[i]
    M_core = M_cores[i]
    
    for fO2_log10_shift in fO2_log10_shifts:
        shift = f"{fO2_log10_shift:.2g}"
        for i in range(len(hmps)):
            hmp = hmps[i]
            hmp_text = f"{hmp:.2g}"
            filename = f"./excel_Uranus_solar_3000K/HOSi/rho{rho_centre}/core_boundary/HOSi_magma_sol_real_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_2.xlsx"
            Pressure = pd.read_excel(filename, sheet_name="state")["pressure"]
            Pressures, Temperatures = Pressure_Temperatures(3000, tau, Pressure[i], 1e7)
            atm_scalings = Pressures/max(Pressures)
            mantle_melt_fraction = 1
            state: ThermodynamicStateProtocol = ThermodynamicState(Temperatures, Pressures)
            init_metallicitys = [1]  # metallicity in x solar units
            

            # Magma - Solubility - Real Gas
        
            H_mass_atm_magma_sol_real = pd.read_excel(filename, sheet_name="element_H")[
                "gas_mass"
            ][i]
            Si_mass_atm_magma_sol_real = pd.read_excel(filename, sheet_name="element_Si")[
                "gas_mass"
            ][i]
            O_mass_atm_magma_sol_real = pd.read_excel(filename, sheet_name="element_O")[
                "gas_mass"
            ][i]
            tot_pressure_magma_sol_real = pd.read_excel(filename, sheet_name="state")["pressure"][
                0
            ]

            mass_constraints = {
                "H": H_mass_atm_magma_sol_real * atm_scalings,
                "O": O_mass_atm_magma_sol_real * atm_scalings,
                "Si": Si_mass_atm_magma_sol_real * atm_scalings,
            }

            model_atm_magma_sol_real.solve(
                state=state,
                mass_constraints=mass_constraints,
                solver_parameters=SolverParameters(multistart=20),
            )
            output_atm_magma_sol_real = model_atm_magma_sol_real.output

            output_atm_magma_sol_real.quick_look()
    
            output_atm_magma_sol_real.to_excel(
                f"./excel_Uranus_solar_3000K/HOSi/rho{rho_centre}/atmosphere/HOSi_atmo_sol_real_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_{hmp_text}Hwt_interH20"
            )
            output_atm_magma_sol_real.to_pickle(
                f"./excel_Uranus_solar_3000K/HOSi/rho{rho_centre}/atmosphere/HOSi_atmo_sol_real_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_{hmp_text}Hwt_interH20"
            )

[00:59:26 - atmodeller.classes             - INFO     ] - species_network = ('H2O_g: IdealGas, NoSolubility', 'H2_g: IdealGas, NoSolubility', 'O2_g: IdealGas, NoSolubility', 'H2O_cd: CondensateActivity, NoSolubility', 'OSi_g: CombinedRealGas, NoSolubility', 'H4Si_g: CombinedRealGas, NoSolubility')
[00:59:26 - atmodeller.classes             - INFO     ] - Thermodynamic data requires temperatures between 200 K and 600 K
[00:59:26 - atmodeller.classes             - INFO     ] - reactions = {0: '1.0 H2O_g = 1.0 H2O_cd',
 1: '2.0 H2O_g = 2.0 H2_g + 1.0 O2_g',
 2: '3.0 H2_g + 1.0 OSi_g = 1.0 H2O_g + 1.0 H4Si_g'}
[00:59:41 - atmodeller.classes             - INFO     ] - Solve (robust) complete: 20 (100.00%) successful model(s)
[00:59:42 - atmodeller.classes             - INFO     ] - Multistart summary: 20 (100.00%) models(s) required 1 attempt(s)
[00:59:42 - atmodeller.classes             - INFO     ] - Solver steps (max) = 40
[00:59:42 - atmodeller.output_core         - DEBUG    ] - Creatin

# HHeCNOSi core-magma

In [10]:
# Initialize EquilibriumModel objects 
model_magma_nosol_ideal = EquilibriumModel(species_HHeCNOSi_magma_nosol_ideal)
model_magma_sol_ideal = EquilibriumModel(species_HHeCNOSi_magma_sol_ideal)
model_magma_sol_real = EquilibriumModel(species_HHeCNOSi_magma_sol_real)

number_of_realisations = 50
mantle_melt_fraction = 1
init_metallicitys = [1]
rho_centres = [3000,3000,3000]
surf_fraction = [0.6]
for i in range(len(rho_centres)):
    rho_centre = rho_centres[i]
    M_core = M_cores[0]
    fugacity_constraints = {O2_rg.name: IronWustiteBuffer(fO2_log10_shifts)}
    for fO2_log10_shift in fO2_log10_shifts:
        shift = f"{fO2_log10_shift:.2g}"
        planet = Planet(
            temperature=3000,
            planet_mass=10*M_earth,
            mantle_melt_fraction=1,
            surface_radius=surf_fraction[i]*R_uranus,
        )
        si_kg_magma = Si_massfraction_palme14 * M_core

        for init_metallicity in init_metallicitys:
            mass_constraints = {
                "H": h_kgs,
                "He": he_kgs_solar,
                "C": init_metallicity * c_kgs_solar,
                "N": init_metallicity * n_kgs_solar,
                "Si": init_metallicity * si_kgs_solar + mantle_melt_fraction * si_kg_magma,
                }
            # Magma - No solubility - Ideal Gas
            model_magma_nosol_ideal.solve(
                state=planet,
                mass_constraints=mass_constraints,
                fugacity_constraints={O2_rg.name: IronWustiteBuffer(np.full(number_of_realisations, fO2_log10_shift))},
                solver_parameters=SolverParameters(multistart=50)
            )
            output_magma_nosol_ideal = model_magma_nosol_ideal.output
            # output_magma_nosol_ideal.quick_look()
            output_magma_nosol_ideal.to_excel(
                f"./excel_Uranus_solar_3000K/HHeCNOSi/rho{rho_centre}/core_boundary/HHeCNOSi_magma_nosol_ideal_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_TEST"
            )
            output_magma_nosol_ideal.to_pickle(
                f"./excel_Uranus_solar_3000K/HHeCNOSi/rho{rho_centre}/core_boundary/HHeCNOSi_magma_nosol_ideal_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_TEST"
            )
            # Magma - Solubility - Ideal Gas
            initial_log_number_moles = output_magma_nosol_ideal.log_number_moles
            model_magma_sol_ideal.solve(
                state=planet,
                mass_constraints=mass_constraints,
                initial_log_number_moles=initial_log_number_moles,
                fugacity_constraints={O2_rg.name: IronWustiteBuffer(np.full(number_of_realisations, fO2_log10_shift))},
                solver_parameters=SolverParameters(multistart=20)
            )
            output_magma_sol_ideal = model_magma_sol_ideal.output
            # output_magma_sol_ideal.quick_look()
            output_magma_sol_ideal.to_excel(
                f"./excel_Uranus_solar_3000K/HHeCNOSi/rho{rho_centre}/core_boundary/HHeCNOSi_magma_sol_ideal_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_TEST"
            )
            output_magma_sol_ideal.to_pickle(
                f"./excel_Uranus_solar_3000K/HHeCNOSi/rho{rho_centre}/core_boundary/HHeCNOSi_magma_sol_ideal_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_TEST"
            )
            # Magma - Solubility - Real Gas
            initial_log_number_moles = output_magma_sol_ideal.log_number_moles
            model_magma_sol_real.solve(
                state=planet,
                mass_constraints=mass_constraints,
                initial_log_number_moles=initial_log_number_moles,
                fugacity_constraints={O2_rg.name: IronWustiteBuffer(np.full(number_of_realisations, fO2_log10_shift))},
                solver_parameters=SolverParameters(multistart=20)
            )
            output_magma_sol_real = model_magma_sol_real.output
            output_magma_sol_real.quick_look()
            output_magma_sol_real.to_excel(
                f"./excel_Uranus_solar_3000K/HHeCNOSi/rho{rho_centre}/core_boundary/HHeCNOSi_magma_sol_real_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_TEST"
            )
            output_magma_sol_real.to_pickle(
                f"./excel_Uranus_solar_3000K/HHeCNOSi/rho{rho_centre}/core_boundary/HHeCNOSi_magma_sol_real_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_TEST"
            )

[00:00:54 - atmodeller.classes             - INFO     ] - species_network = ('H2O_g: IdealGas, NoSolubility', 'H2_g: IdealGas, NoSolubility', 'O2_g: IdealGas, NoSolubility', 'OSi_g: IdealGas, NoSolubility', 'H4Si_g: IdealGas, NoSolubility', 'O2Si_cd: CondensateActivity, NoSolubility', 'CO2_g: IdealGas, NoSolubility', 'CO_g: IdealGas, NoSolubility', 'CH4_g: IdealGas, NoSolubility', 'N2_g: IdealGas, NoSolubility', 'H3N_g: IdealGas, NoSolubility', 'He_g: IdealGas, NoSolubility')
[00:00:54 - atmodeller.classes             - INFO     ] - Thermodynamic data requires temperatures between 200 K and 6000 K
[00:00:54 - atmodeller.classes             - INFO     ] - reactions = {0: '2.0 H2_g + 0.5 O2Si_cd = 1.0 H2O_g + 0.5 H4Si_g',
 1: '0.5 H4Si_g + 1.0 CO2_g = 1.0 H2_g + 0.5 O2Si_cd + 1.0 CO_g',
 2: '1.0 H4Si_g + 1.0 CO2_g = 1.0 O2Si_cd + 1.0 CH4_g',
 3: '0.5 H4Si_g + 0.5 O2Si_cd = 1.0 H2_g + 1.0 OSi_g',
 4: '1.5 H2_g + 0.5 N2_g = 1.0 H3N_g',
 5: '2.0 H2_g + 1.0 O2Si_cd = 1.0 O2_g + 1.0 H4Si_g'}


EquinoxRuntimeError: Above is the stack outside of JIT. Below is the stack inside of JIT:
  File "/home/mathias/atmodeller/atmodeller/solvers.py", line 335, in solve_with_jit
    multi_sol = lax.cond(
                ^^^^^^^^^
  File "/home/mathias/atmodeller/atmodeller/solvers.py", line 337, in <lambda>
    lambda _: solve_with_stability_multistart(key),  # True: Use stability solver
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/mathias/atmodeller/atmodeller/solvers.py", line 321, in solve_with_stability_multistart
    return tau_sweep_solver(base_solution_array, parameters, subkey)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/mathias/atmodeller/atmodeller/solvers.py", line 261, in tau_sweep_solver
    _, results = jax.lax.scan(solve_tau_step, initial_carry, tau_array)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/mathias/atmodeller/atmodeller/solvers.py", line 225, in solve_tau_step
    n

In [44]:
for fO2_log10_shift in fO2_log10_shifts:
    shift = f"{fO2_log10_shift:.2g}"
    for i in range(1):
        hmp = hmps[0]
        filename = f"./excel_Uranus_solar_3000K/HHeCNOSi/rho{rho_centre}/core_boundary/HHeCNOSi_magma_sol_real_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_2.xlsx"
        Pressure = pd.read_excel(filename, sheet_name="state")["pressure"]
        Pressures, Temperatures = Pressure_Temperatures(3000, tau, Pressure[i], 1e7)
        Pressures = Pressures*10**(-4)
        for i in range(len(Temperatures)):
            result = get_phase(Temperatures[i], Pressures[i])
            print(f"{result['phase']:20s} "
              f"(ρ={result['density']:8.1f} kg/m³) "
              f"[closest point: T={result['actual_T']:.0f}K, P={result['actual_P']:.3f}GPa]")
        print(list_phases(Temperatures, Pressures))

Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=100K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=100K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=100K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=100K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=100K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=100K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=100K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=100K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=100K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=100K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=100K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=100K, P=0.000GPa]
Ice Ih               (ρ=   933.0 kg/m³) [closest point: T=100K, P=0.000GPa]
Ice Ih      

# HHeCNOSi atmosphere

In [16]:
init_metallicity = 1
mantle_melt_fraction = 1
rho_centres = [3000]
model_atm_magma_sol_real = EquilibriumModel(species_HHeCNOSi_atmosphere)
for i in range(len(rho_centres)):
    rho_centre = rho_centres[i]
    M_core = M_cores[i]
    
    for fO2_log10_shift in fO2_log10_shifts:
        shift = f"{fO2_log10_shift:.2g}"
        for i in range(len(hmps)):
            hmp = hmps[i]
            hmp_text = f"{hmp:.2g}"
            filename = f"./excel_Uranus_solar_3000K/HHeCNOSi/rho{rho_centre}/core_boundary/HHeCNOSi_magma_sol_real_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_2.xlsx"
            Pressure = pd.read_excel(filename, sheet_name="state")["pressure"]
            Pressures, Temperatures = Pressure_Temperatures(3000, tau, Pressure[i], 1e7)
            atm_scalings = Pressures/max(Pressures)
            mantle_melt_fraction = 1
            state: ThermodynamicStateProtocol = ThermodynamicState(Temperatures, Pressures)
            init_metallicitys = [1]  # metallicity in x solar units
            
            # Magma - Solubility - Real Gas
        
            H_mass_atm_magma_sol_real = pd.read_excel(filename, sheet_name="element_H")[
                "gas_mass"
            ][i]
            Si_mass_atm_magma_sol_real = pd.read_excel(filename, sheet_name="element_Si")[
                "gas_mass"
            ][i]
            O_mass_atm_magma_sol_real = pd.read_excel(filename, sheet_name="element_O")[
                "gas_mass"
            ][i]
            He_mass_atm_magma_sol_real = pd.read_excel(filename, sheet_name="element_He")[
                "gas_mass"
            ][i]
            C_mass_atm_magma_sol_real = pd.read_excel(filename, sheet_name="element_C")[
                "gas_mass"
            ][i]
            N_mass_atm_magma_sol_real = pd.read_excel(filename, sheet_name="element_N")[
                "gas_mass"
            ][i]
            tot_pressure_magma_sol_real = pd.read_excel(filename, sheet_name="state")["pressure"][
                0
            ]

            mass_constraints = {
                "H": H_mass_atm_magma_sol_real * atm_scalings,
                "O": O_mass_atm_magma_sol_real * atm_scalings,
                "Si": Si_mass_atm_magma_sol_real * atm_scalings,
                "C": C_mass_atm_magma_sol_real * atm_scalings,
                "N": N_mass_atm_magma_sol_real * atm_scalings,
                "He": He_mass_atm_magma_sol_real * atm_scalings,
            }

            model_atm_magma_sol_real.solve(
                state=state,
                mass_constraints=mass_constraints,
                solver_parameters=SolverParameters(multistart=20),
            )
            output_atm_magma_sol_real = model_atm_magma_sol_real.output

            output_atm_magma_sol_real.quick_look()

            output_atm_magma_sol_real.to_excel(
                f"./excel_Uranus_solar_3000K/HHeCNOSi/rho{rho_centre}/atmosphere/HHeCNOSi_atmo_sol_real_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_{hmp_text}Hwt_condensedH2O"
            )
            output_atm_magma_sol_real.to_pickle(
                f"./excel_Uranus_solar_3000K/HHeCNOSi/rho{rho_centre}/atmosphere/HHeCNOSi_atmo_sol_real_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_{hmp_text}Hwt_condensedH2O"
            )

[13:10:43 - atmodeller.classes             - INFO     ] - species_network = ('H2O_g: CombinedRealGas, NoSolubility', 'H2_g: CombinedRealGas, NoSolubility', 'O2_g: CombinedRealGas, NoSolubility', 'OSi_g: CombinedRealGas, NoSolubility', 'H4Si_g: CombinedRealGas, NoSolubility', 'H2O_cd: CondensateActivity, NoSolubility', 'C_cd: CondensateActivity, NoSolubility', 'CSi_cd: CondensateActivity, NoSolubility', 'Si_cd: CondensateActivity, NoSolubility', 'N4Si3_cd: CondensateActivity, NoSolubility', 'CO2_g: CombinedRealGas, NoSolubility', 'CO_g: CombinedRealGas, NoSolubility', 'CH4_g: CombinedRealGas, NoSolubility', 'N2_g: CombinedRealGas, NoSolubility', 'H3N_g: RedlichKwong49, NoSolubility', 'He_g: CombinedRealGas, SolubilityPowerLaw')
[13:10:43 - atmodeller.classes             - INFO     ] - Thermodynamic data requires temperatures between 200 K and 600 K
[13:10:43 - atmodeller.classes             - INFO     ] - reactions = {0: '1.0 H2O_cd = 1.0 H2O_g',
 1: '1.0 H4Si_g + 1.0 C_cd = 2.0 H2_g + 